<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [30]:
from functools import lru_cache

import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split


@lru_cache(maxsize=1)
def _fetch_openml_dataset():
    """Baixa o dataset uma vez e reutiliza em chamadas seguintes."""
    candidates = [
        ("Fashion-MNIST", {"version": 1}),
        ("mnist_784", {}),
    ]

    last_error = None
    for name, extra_kwargs in candidates:
        try:
            X, y = fetch_openml(
                name=name,
                as_frame=False,
                parser="auto",
                return_X_y=True,
                **extra_kwargs,
            )
            return X.astype(np.float32), y.astype(np.int64), name
        except Exception as exc:
            last_error = exc

    raise RuntimeError(
        "Nao foi possivel carregar o dataset via OpenML. "
        f"Ultimo erro: {last_error}"
    )


def load_data(seed=42, test_size=0.2, max_samples=15000, normalize=False):
    """Carrega dados, aplica amostragem opcional e divide em treino/teste."""
    X, y, _ = _fetch_openml_dataset()

    if max_samples is not None and max_samples < len(X):
        X, _, y, _ = train_test_split(
            X,
            y,
            train_size=max_samples,
            stratify=y,
            random_state=seed,
        )

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        stratify=y,
        random_state=seed,
    )

    if normalize:
        X_train = X_train / 255.0
        X_test = X_test / 255.0

    return X_train, X_test, y_train, y_test

**Resposta (Questão 1):**
Para carregar os dados e formatar as devidas fatias, implementamos a função de particionamento `load_data` delegando as definições estocásticas para o controle de `seed`. O uso da normalização de dados para esse modelo não é mandatório, embora possa auxiliar na organização. Modelos focados em Árvores atuam baseando as decisões unicamente no ordenamento relativo das variáveis contínuas (e não com as distâncias), garantindo o sucesso perante conjuntos de proporções discrepantes.

# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [31]:
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier


def train_random_forest(
    X_train,
    y_train,
    seed=42,
    n_estimators=200,
    max_depth=None,
    n_jobs=-1,
    **kwargs,
):
    """Treina Random Forest com controle total de aleatoriedade."""
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=seed,
        n_jobs=n_jobs,
        **kwargs,
    )
    model.fit(X_train, y_train)
    return model


def train_adaboost(
    X_train,
    y_train,
    seed=42,
    n_estimators=80,
    learning_rate=0.7,
    base_depth=2,
    **kwargs,
):
    """Treina AdaBoost com estimador base raso para ganho de desempenho."""
    base_estimator = DecisionTreeClassifier(max_depth=base_depth, random_state=seed)

    # Compatibilidade com diferentes versoes do scikit-learn.
    model_kwargs = dict(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        random_state=seed,
    )
    model_kwargs.update(kwargs)

    if "estimator" in AdaBoostClassifier().get_params():
        model_kwargs["estimator"] = base_estimator
    else:
        model_kwargs["base_estimator"] = base_estimator

    model = AdaBoostClassifier(**model_kwargs)
    model.fit(X_train, y_train)
    return model

**Resposta (Questão 2):**
As funções `train_random_forest` e `train_adaboost` foram projetadas para instanciar as respectivas classes do Sklearn. Importante notar o repasse rígido de `random_state=seed` para ambos as chamadas, mantendo as árvores passíveis de testagem previsível sob rodadas iterativas, já que lidam por padrão com divisões probabilísticas ou escolhas ponderadas.

# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [32]:
def evaluate(model, X_test, y_test, average="macro", return_metrics=False):
    """Avalia o modelo com multiplas metricas e retorno flexivel."""
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average=average, zero_division=0
    )

    metrics = {
        "accuracy": float(acc),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }
    return metrics if return_metrics else metrics["accuracy"]

**Resposta (Questão 3):**
A função implementada recebe como parâmetros os modelos e seus correspondentes X de testes. Usando predições com o método `predict`, alcançamos as instâncias que compõem o sumário final e derivamos de relance as notas avaliativas primárias, dentre elas e com foco mandatário no escopo, está presente a averiguação generalista global (acurácia). Além disso, exportações paramétricas avançadas incluem estatísticas sobre variâncias menores (precisão, *recall*, formatações em f-score).

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [33]:
def run_pipeline(
    model_type="rf",
    seed=42,
    normalize=False,
    max_samples=15000,
    return_metrics=False,
    **model_kwargs,
):
    """Executa o pipeline completo para RF ou AdaBoost."""
    X_train, X_test, y_train, y_test = load_data(
        seed=seed, normalize=normalize, max_samples=max_samples
    )

    model_key = model_type.lower()
    if model_key == "rf":
        model = train_random_forest(X_train, y_train, seed=seed, **model_kwargs)
    elif model_key == "ab":
        model = train_adaboost(X_train, y_train, seed=seed, **model_kwargs)
    else:
        raise ValueError("model_type deve ser 'rf' ou 'ab'.")

    metrics = evaluate(model, X_test, y_test, return_metrics=True)
    return metrics if return_metrics else metrics["accuracy"]

**Resposta (Questão 4):**
A formulação agrupa a utilidade das etapas em `run_pipeline`, simplificando as etapas modulares elaboradas de forma paralela outrora. Uma amostra das decisões iniciais flui das configurações da função passando pelo treinamento contido até desaguar finalmente com a nota unificada e mensurável atestando o funcionamento. Todos os processos recebem em cadeia o mesmo `seed` raiz, e aceita configurações arbitrárias enviadas pelos parâmetros elásticos extras (kwargs), mantendo flexibilidade para invocações de comparação.

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

In [34]:
import pandas as pd


def compare_models(seed=42, max_samples=15000):
    """Compara RF e AdaBoost com as metricas solicitadas."""
    rows = []
    for model_type in ["rf", "ab"]:
        metrics = run_pipeline(
            model_type=model_type,
            seed=seed,
            max_samples=max_samples,
            return_metrics=True,
        )
        metrics["model"] = "Random Forest" if model_type == "rf" else "AdaBoost"
        rows.append(metrics)

    result = pd.DataFrame(rows)[["model", "accuracy", "precision", "recall", "f1"]]
    return result.sort_values(by="f1", ascending=False).reset_index(drop=True)


def best_initial_model(seed=42, max_samples=15000):
    table = compare_models(seed=seed, max_samples=max_samples)
    return table.iloc[0]["model"], table

**Resposta (Questão 5):**
O **Random Forest** apresentou um desempenho inicial significativamente melhor que o AdaBoost, alcançando acurácia e F1-Score próximos de 87%, enquanto o AdaBoost ficou restrito a patamares próximos de 64% com os parâmetros base da simulação inicial.

# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

In [35]:
def compare_seeds(seeds=(42, 7), model_types=("rf", "ab"), max_samples=15000):
    """Avalia estabilidade do experimento para diferentes sementes."""
    rows = []
    for seed in seeds:
        for model_type in model_types:
            metrics = run_pipeline(
                model_type=model_type,
                seed=seed,
                max_samples=max_samples,
                return_metrics=True,
            )
            rows.append(
                {
                    "seed": seed,
                    "model": model_type,
                    "accuracy": metrics["accuracy"],
                    "precision": metrics["precision"],
                    "recall": metrics["recall"],
                    "f1": metrics["f1"],
                }
            )
    return pd.DataFrame(rows).sort_values(["model", "seed"]).reset_index(drop=True)


def reproducibility_check(model_type="rf", seed=42):
    acc1 = run_pipeline(model_type=model_type, seed=seed)
    acc2 = run_pipeline(model_type=model_type, seed=seed)
    return {
        "first_run": acc1,
        "second_run": acc2,
        "absolute_diff": abs(acc1 - acc2),
    }

**Resposta (Questão 6):**
As métricas **mudam ligeiramente** de acordo com a `seed` por causa das variáveis aleatórias subjacentes no momento da partição dos dados de treino/teste e na injeção natural de estocasticidade de cada estimador de base.
Contudo, o experimento é **rotineiramente reprodutível**, já que a manutenção de uma `seed` fixa reencena a mesmíssima modelagem nas subamostragens, entregando resultados perfeitamente idênticos toda vez.

# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [36]:
def overfitting_report(seed=42, max_samples=15000):
    """Compara treino vs teste para identificar overfitting nos modelos."""
    X_train, X_test, y_train, y_test = load_data(seed=seed, max_samples=max_samples)

    rf = train_random_forest(X_train, y_train, seed=seed, n_estimators=250)
    ab = train_adaboost(X_train, y_train, seed=seed, n_estimators=120)

    rows = []
    for name, model in [("Random Forest", rf), ("AdaBoost", ab)]:
        train_acc = float(model.score(X_train, y_train))
        test_acc = float(model.score(X_test, y_test))
        rows.append(
            {
                "model": name,
                "train_accuracy": train_acc,
                "test_accuracy": test_acc,
                "generalization_gap": train_acc - test_acc,
            }
        )
    return pd.DataFrame(rows).sort_values("generalization_gap", ascending=False)

**Resposta (Questão 7):**
**- Existe overfitting?** Sim, existe um overfitting evidenciado pela diferença expressiva de pontuação das predições sobre os dados de validação contra a perfeição atingida na base de treino.
**- Qual modelo sofre mais?** O modelo com propensão clara a memorizar excessivamente o conjunto e demonstrar lacunas de generalização (gap) consideráveis frente a novos dados é o **Random Forest**, por depender do aumento da aleatoriedade e limitação de profundidade de nó para suavizar essas discrepâncias.

# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

In [37]:
def hyperparameter_sensitivity(seed=42, max_samples=15000):
    """Varia n_estimators em RF e AdaBoost e compara impacto no desempenho."""
    rf_estimators = [50, 100, 200]
    ab_estimators = [30, 60, 120]

    rows = []
    for n in rf_estimators:
        metrics = run_pipeline(
            model_type="rf",
            seed=seed,
            max_samples=max_samples,
            return_metrics=True,
            n_estimators=n,
        )
        rows.append({"model": "rf", "n_estimators": n, **metrics})

    for n in ab_estimators:
        metrics = run_pipeline(
            model_type="ab",
            seed=seed,
            max_samples=max_samples,
            return_metrics=True,
            n_estimators=n,
        )
        rows.append({"model": "ab", "n_estimators": n, **metrics})

    return pd.DataFrame(rows).sort_values(["model", "n_estimators"]).reset_index(drop=True)

**Resposta (Questão 8):**
Ao variar os `n_estimators` (o número de estimadores base), o desempenho **muda significativamente nas fases iniciais**. No entanto, a sensibilidade é distinta entre os algoritmos testados:
O modelo **AdaBoost costuma ser mais sensível a mudanças constantes** e exige maior controle combinado com `learning_rate` na busca por uma maior eficiência sequencial. Em contrapartida, o Random Forest ganha acurácia ao agregar as primeiras dezenas de árvores, mas gradualmente estaciona as suas melhoras em um platô a partir de determinado ponto, mitigando o retorno obtido a cada nova árvore incrementada.

# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

**Resposta (Questão 9):**
**1.** *Não.* Avaliar os resultados valendo-se única e exclusivamente da acurácia global tende a dissimular ineficiências em instâncias ou classes específicas do *dataset*, camuflando potenciais desbalanceamentos subjacentes. As métricas acopladas – Precisão, Recall e F1-Score – agem com maior amplitude, confirmando ou negando um desempenho real qualitativo.
**2.** A garantia da fidedignidade é estendida quando fixamos o `random_state` por todas as frentes com chamadas sujeitas à randomização. Isso estabiliza as amostragens do teste atual contra si próprio. Variar o valor dessas *seeds* também atua como validador de sanidade, provando estatisticamente que a avaliação não superestima bons frutos provenientes de um corte excessivamente enviesado dos dados temporários.
**3.** (i) Depender apenas de uma retenção singular para as previsões dita as generalizações com base em somente uma fatia de teste, mascarando outras nuances na amostragem; (ii) Configurar testes dinâmicos ajustando continuamente os parâmetros orientados pelos resultados obtidos leva ao viés retrospectivo, inflando um subentendido e danoso *overfitting metodológico*.
**4.** O *pipeline* exibe uma modularidade *parcialmente confiável*: a reprodutibilidade está assegurada, e as métricas coletadas propõem transparência ao invés de olhar de túnel. Todavia, ressentimos a não elaboração de uma matriz com Validações Cruzadas (Cross Validation) e a ausência de isolamento real para dados em um Conjunto de Validação restrito às sintonias dinâmicas.

In [ ]:
# Célula para visualização dos resultados
try:
    from IPython.display import display
except ImportError:
    display = print

print("="*50)
print("COMPARAÇÃO DE MODELOS (Questão 3, 4, 6, 8)")
print("="*50)
df_models = compare_models()
display(df_models)

print("\n" + "="*50)
print("COMPARAÇÃO DE DIFERENTES SEEDS (Questão 5)")
print("="*50)
df_seeds = compare_seeds()
display(df_seeds)

print("\n" + "="*50)
print("ANÁLISE DE OVERFITTING - RANDOM FOREST (Questão 7)")
print("="*50)
df_overfitting = overfitting_report()
display(df_overfitting)

print("\n" + "="*50)
print("SENSIBILIDADE DE HIPERPARÂMETROS - ADABOOST (Questão 9)")
print("="*50)
df_tuning = hyperparameter_sensitivity()
display(df_tuning)

COMPARAÇÃO DE MODELOS (Questão 3, 4, 6, 8)


,model,accuracy,precision,recall,f1
0,Random Forest,0.871000,0.869833,0.871000,0.868953
1,AdaBoost,0.643333,0.660635,0.643333,0.619798



COMPARAÇÃO DE DIFERENTES SEEDS (Questão 5)


,seed,model,accuracy,precision,recall,f1
0,7,ab,0.676000,0.688144,0.676000,0.675010
1,42,ab,0.643333,0.660635,0.643333,0.619798
2,7,rf,0.854667,0.852223,0.854667,0.852452
3,42,rf,0.871000,0.869833,0.871000,0.868953



ANÁLISE DE OVERFITTING - RANDOM FOREST (Questão 7)


,model,train_accuracy,test_accuracy,generalization_gap
0,Random Forest,1.000000,0.871333,0.128667
1,AdaBoost,0.640917,0.637000,0.003917



SENSIBILIDADE DE HIPERPARÂMETROS - ADABOOST (Questão 9)


,model,n_estimators,accuracy,precision,recall,f1
0,ab,30,0.686333,0.706372,0.686333,0.686175
1,ab,60,0.671333,0.687632,0.671333,0.667298
2,ab,120,0.637000,0.631830,0.637000,0.611298
3,rf,50,0.866000,0.864813,0.866000,0.863860
4,rf,100,0.870000,0.868903,0.870000,0.867566
5,rf,200,0.871000,0.869833,0.871000,0.868953
